# 01 · Carga de datos Excel → Supabase
Manufactura textil moda Colombia · 89 tiendas · 3 años

### Antes de correr
1. `.env` con `SUPABASE_URL` y `SUPABASE_ANON_KEY`
2. Tablas creadas en Supabase (`sql/01_crear_tablas.sql`)
3. Archivos Excel en `datos_excel/`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','pandas','openpyxl','python-dotenv','supabase'])
print('OK')

In [ ]:
import pandas as pd, sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))
from conexion import get_supabase_client, get_datos_dir, df_to_supabase
sb = get_supabase_client()
DIR = get_datos_dir()
print('Datos en:', DIR)

## 1 · Dimensiones

In [ ]:
df_tiendas = pd.read_excel(DIR/'dim_tiendas.xlsx')
df_tipos   = pd.read_excel(DIR/'dim_tipos_producto.xlsx')
df_eventos = pd.read_excel(DIR/'dim_eventos.xlsx')
df_eventos['fecha'] = pd.to_datetime(df_eventos['fecha']).dt.date
print(f'Tiendas:{len(df_tiendas)} Tipos:{len(df_tipos)} Eventos:{len(df_eventos)}')
assert df_tiendas['tienda_id'].nunique()==len(df_tiendas)
print('Validaciones OK')

In [ ]:
# Limpiar y cargar dimensiones
sb.table('dim_eventos').delete().neq('evento_id',0).execute()
sb.table('dim_tipos_producto').delete().neq('tipo_producto','').execute()
sb.table('dim_tiendas').delete().neq('tienda_id',0).execute()
print('Tablas limpiadas')
df_tiendas2 = df_tiendas.drop(columns=['tienda_id'],errors='ignore')
df_tiendas2['fecha_apertura'] = pd.to_datetime(df_tiendas2['fecha_apertura']).dt.date.astype(str)
df_tiendas2['activa'] = df_tiendas2['activa'].astype(bool)
df_to_supabase(sb, df_tiendas2, 'dim_tiendas')
df_to_supabase(sb, df_tipos, 'dim_tipos_producto')
df_eventos2 = df_eventos.drop(columns=['evento_id'],errors='ignore')
df_to_supabase(sb, df_eventos2, 'dim_eventos')

## 2 · Ventas 2022–2024

In [ ]:
def limpiar_ventas(df):
    df = df.copy()
    df['fecha'] = pd.to_datetime(df['fecha']).dt.date.astype(str)
    df = df[df['unidades_vendidas']>0]
    df = df.dropna(subset=['fecha','tienda_id','tipo_producto','unidades_vendidas','valor_venta'])
    if 'es_precio_pleno' not in df.columns:
        df['es_precio_pleno'] = df['descuento_pct']==0
    df['es_precio_pleno'] = df['es_precio_pleno'].astype(bool)
    df['tienda_id'] = df['tienda_id'].astype(int)
    df = df.drop(columns=['venta_id'],errors='ignore')
    return df

sb.table('fact_ventas_diarias').delete().neq('venta_id',0).execute()
print('fact_ventas_diarias limpiada')
for anio in [2022,2023,2024]:
    df_v = limpiar_ventas(pd.read_excel(DIR/f'ventas_{anio}.xlsx'))
    print(f'ventas_{anio}: {len(df_v):,} filas | ${df_v["valor_venta"].sum():,.0f} COP')
    df_to_supabase(sb, df_v, 'fact_ventas_diarias')

## 3 · Inventario semanal

In [ ]:
sb.table('fact_inventario_semanal').delete().neq('inv_id',0).execute()
print('fact_inventario_semanal limpiada')
for parte in [1,2]:
    df_i = pd.read_excel(DIR/f'inventario_semanal_parte{parte}.xlsx')
    df_i['fecha'] = pd.to_datetime(df_i['fecha']).dt.date.astype(str)
    df_i['tienda_id'] = df_i['tienda_id'].astype(int)
    df_i = df_i.drop(columns=['inv_id'],errors='ignore')
    print(f'inventario parte {parte}: {len(df_i):,} filas')
    df_to_supabase(sb, df_i, 'fact_inventario_semanal')

## 4 · Verificación

In [ ]:
print('=== RESUMEN SUPABASE ===')
for t in ['dim_tiendas','dim_tipos_producto','dim_eventos','fact_ventas_diarias','fact_inventario_semanal']:
    r = sb.table(t).select('*', count='exact').limit(1).execute()
    print(f'  {t:<35} {r.count:>10,}')
print('\nOK - Correr notebook 02')